In [ ]:
# import libraries
import math
import os
import pickle
import random
import time
import itertools
import importlib

# import numpy
import numpy as np
import matplotlib.pyplot as plt
import sklearn.preprocessing
from sklearn import preprocessing

import torch
import scipy
import scipy.sparse as sparse
import scipy.io

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# import Koopman Libraries
from core.koopman_core_linear_GPU import KoopDNN_linear, KoopmanNet_linear, KoopmanNetCtrl_linear
from core.util import fit_standardizer
from models.koop_model import model_matricies, lift_raw, lift_scaled

# ===== import Frenet-real-state dynamics file =====
import dynamics.cacalv3 as cacalv3

importlib.reload(cacalv3)
from dynamics.cacalv3 import FK_solver, single_vehicle_data_gen_multi

# control
from control_files.nmpc_osqp_adapt_four import NonlinearMPCController
from dynamics.learned_models_control.linear_dynamics_four import linear_Dynamics


# ===== helper for Frenet path =====
def build_test_frenet_path_from_xy(x_ref_path, y_ref_path):
    dx_seg = np.diff(x_ref_path)
    dy_seg = np.diff(y_ref_path)
    ds_seg = np.sqrt(dx_seg ** 2 + dy_seg ** 2)
    s_ref = np.concatenate(([0.0], np.cumsum(ds_seg)))

    dx = np.gradient(x_ref_path)
    dy = np.gradient(y_ref_path)
    psi_ref = np.unwrap(np.arctan2(dy, dx))

    ds = np.gradient(s_ref)
    curvature_ref = np.gradient(psi_ref) / np.maximum(ds, 1e-8)
    curvature_ref = np.nan_to_num(curvature_ref)

    return s_ref, psi_ref, curvature_ref


# ===== Frenet -> Cartesian for plotting =====
def frenet_to_global(s_arr, ey_arr, s_ref_path, x_ref_path, y_ref_path, psi_ref_path):
    x_center = np.interp(s_arr, s_ref_path, x_ref_path)
    y_center = np.interp(s_arr, s_ref_path, y_ref_path)
    psi_center = np.interp(s_arr, s_ref_path, psi_ref_path)

    x_global = x_center - ey_arr * np.sin(psi_center)
    y_global = y_center + ey_arr * np.cos(psi_center)
    return x_global, y_global


def to_numpy(x):
    if hasattr(x, "detach"):
        return x.detach().cpu().numpy()
    return np.array(x)


import matplotlib.pyplot as plt
import matplotlib as mpl

# ===== 修复中文乱码 =====
mpl.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']  # 优先使用中文字体
mpl.rcParams['axes.unicode_minus'] = False


# ===== 修复 clip 导致的初始状态归零问题 =====
def clip_closed_loop_state(x, s_upper=500.0, s_lower=-30.0):
    x = np.asarray(x, dtype=float).copy()
    x[0] = np.clip(x[0], s_lower, s_upper)  # s (允许为负数，适配后方跟随车辆)
    x[1] = np.clip(x[1], -8.0, 8.0)  # e_y (放宽横向偏差限制)
    x[2] = np.clip(x[2], -1.2, 1.2)  # e_psi
    x[3] = np.clip(x[3], 0.0, 10.0)  # v_x
    x[4] = np.clip(x[4], -4.0, 4.0)  # v_y
    x[5] = np.clip(x[5], -3.0, 3.0)  # r
    return x


def is_finite_vector(x):
    x = np.asarray(x, dtype=float)
    return np.all(np.isfinite(x))


def safe_FK_step(x, u, pars, SNR_DB, i, fallback_state=None, verbose=False):
    """
    对 FK_solver 做保护：
    - 若求解结果非有限，则回退到 fallback_state 或当前状态
    - 输出统一做 clip
    """
    try:
        x_next = FK_solver(x, u, pars, sensor_noise=False, SNR_DB=SNR_DB, i=i)
    except Exception as e:
        if verbose: print(f"FK_solver exception at step {i}: {e}")
        return clip_closed_loop_state(fallback_state if fallback_state is not None else x)

    if not is_finite_vector(x_next):
        if verbose: print(f"Non-finite FK_solver output at step {i}")
        return clip_closed_loop_state(fallback_state if fallback_state is not None else x)

    return clip_closed_loop_state(x_next)


In [ ]:

# =============================================================================
# 系统拟合与数据加载 (复用你训练好的模型)
# =============================================================================
linear = True

# 参数设定 (必须与训练时一致)
num_states = 6
num_inputs = 2
dt = 0.02
vx_nom = 3.0

sys_pars_base = {}
sys_pars_base = {
    "num_states": num_states, "num_inputs": num_inputs, "dt": dt,
    "m": 1500.0, "Iz": 2250.0, "lf": 1.2, "lr": 1.6,
    "Cf": 80000.0, "Cr": 80000.0, "curvature_ref": 0.0,
    "uncertainty": "NA", "amp": 0.0, "seed": 42
}
sys_pars_new = dict(sys_pars_base)
sys_pars = dict(sys_pars_base)

SNR_DB = 30
state_labels = ["s", "e_y", "e_psi", "v_x", "v_y", "r"]
input_labels = ["delta", "a_x"]

# ===== 载入模型与 Standardizer =====
# 假设你已经跑过前面的训练代码，这里直接载入模型和归一化器
# 如果是在同一个 Jupyter Notebook 里，这些变量应该在内存中：
# model_koop_dnn_lin, standardizer_x_kdnn, standardizer_u_kdnn, net_params_lin

net = model_koop_dnn_lin.net
A_lin = to_numpy(net.A.weight)
B_lin = to_numpy(net.B.weight)
C_lin = to_numpy(net.C.weight)

# A矩阵稳定化投影
eigvals, eigvecs = np.linalg.eig(A_lin)
eigvals_proj = np.array([ev if np.abs(ev) <= 0.999 else ev / np.abs(ev) * 0.999 for ev in eigvals])
A_lin_stable = np.real(eigvecs @ np.diag(eigvals_proj) @ np.linalg.inv(eigvecs))

first_obs_const = int(net_params_lin["first_obs_const"])
override_C = net_params_lin["override_C"]
n_obs_lin = int(net_params_lin["encoder_output_dim"]) + num_states + first_obs_const if override_C else int(
    net_params_lin["encoder_output_dim"]) + first_obs_const

# ====================== 四车矩形编队参数 ======================
NUM_VEHICLES = 4

# 矩形队形期望间距（单位：米）
DESIRED_LONGITUDINAL_GAP = 8.0  # 前后车纵向间距
DESIRED_LATERAL_GAP = 4.0  # 左右车横向间距

# 初始队形相对 Leader 的偏移 [Δs, Δe_y]
formation_offsets = np.array([
    [0.0, 0.0],  # Vehicle 0: Leader
    [0.0, DESIRED_LATERAL_GAP],  # Vehicle 1: 右前方
    [-DESIRED_LONGITUDINAL_GAP, 0.0],  # Vehicle 2: 左后方
    [-DESIRED_LONGITUDINAL_GAP, DESIRED_LATERAL_GAP]  # Vehicle 3: 右后方
])

Q_formation_weight = 800.0  # 队形保持额外惩罚权重

In [ ]:

# =============================================================================
# Testing - 100m Double Lane Change Baseline
# =============================================================================

# =========================
# Reference path generation
# =========================
path_length = 100.0
T_traj = path_length / vx_nom
t_ref = np.arange(0, T_traj + dt, dt)
traj_length = t_ref.size
x_path = vx_nom * t_ref

y_ref_path = (0.4 * np.exp(-0.5 * ((x_path - 30.0) / 16.0) ** 2)
              - 0.4 * np.exp(-0.5 * ((x_path - 70.0) / 16.0) ** 2))

s_ref_path, psi_ref_path, curvature_ref_path = build_test_frenet_path_from_xy(x_path, y_ref_path)

x_ref_raw = np.zeros((num_states, traj_length))
x_ref_raw[0, :] = s_ref_path
x_ref_raw[1, :] = 0.0
x_ref_raw[2, :] = 0.0
x_ref_raw[3, :] = vx_nom
x_ref_raw[4, :] = 0.0
x_ref_raw[5, :] = vx_nom * curvature_ref_path

x_ref_scaled = standardizer_x_kdnn.transform(x_ref_raw.T).T


def generate_formation_reference(x_ref_scaled, formation_offsets, standardizer_x):
    """生成四车带偏移的参考轨迹（scaled空间）"""
    num_states_total = num_states * NUM_VEHICLES
    x_ref_formation = np.zeros((num_states_total, x_ref_scaled.shape[1]))
    for v in range(NUM_VEHICLES):
        offset_s = formation_offsets[v, 0]
        offset_ey = formation_offsets[v, 1]
        idx = v * num_states
        x_ref_formation[idx:idx + num_states, :] = x_ref_scaled.copy()
        # 缩放空间下的偏移（正确做法：除以对应标准差）
        x_ref_formation[idx + 0, :] += offset_s / standardizer_x.scale_[0]
        x_ref_formation[idx + 1, :] += offset_ey / standardizer_x.scale_[1]
    return x_ref_formation


x_ref_formation = generate_formation_reference(x_ref_scaled, formation_offsets, standardizer_x_kdnn)

In [ ]:

# =========================
# MPC 权重与约束配置
# =========================
# 单车基础权重
Q_mpc_lin_noadapt = sparse.diags([0.0, 1800.0, 600.0, 8.0, 1.0, 1.0])
QN_mpc_lin_noadapt = sparse.diags([0.0, 2500.0, 1200.0, 10.0, 1.0, 1.0])
R_mpc_lin_noadapt = sparse.diags([120.0, 12.0])

xmin_lin_noadapt = standardizer_x_kdnn.transform(
    np.array([-10.0, -5.0, -1.2, 0.5, -4.0, -3.0]).reshape(1, -1)).flatten()
xmax_lin_noadapt = standardizer_x_kdnn.transform(np.array([500.0, 5.0, 1.2, 8.0, 4.0, 3.0]).reshape(1, -1)).flatten()
umin_lin_noadapt = np.array([-0.08, -1.0])
umax_lin_noadapt = np.array([0.08, 1.0])

# --- 构建带有队形耦合的 Q 矩阵 (Laplacian 结构) ---
nx = Q_mpc_lin_noadapt.shape[0]
Q_F_base = sparse.diags([Q_formation_weight, Q_formation_weight, 0.0, 0.0, 0.0, 0.0])

Q_blocks = [[None] * NUM_VEHICLES for _ in range(NUM_VEHICLES)]
QN_blocks = [[None] * NUM_VEHICLES for _ in range(NUM_VEHICLES)]
for i in range(NUM_VEHICLES):
    for j in range(NUM_VEHICLES):
        if i == j:
            coeff = (NUM_VEHICLES - 1) if i == 0 else 1
            Q_blocks[i][j] = Q_mpc_lin_noadapt + coeff * Q_F_base
            QN_blocks[i][j] = QN_mpc_lin_noadapt + coeff * Q_F_base
        elif i == 0 or j == 0:
            Q_blocks[i][j] = -Q_F_base
            QN_blocks[i][j] = -Q_F_base
        else:
            Q_blocks[i][j] = sparse.csc_matrix((nx, nx))
            QN_blocks[i][j] = sparse.csc_matrix((nx, nx))

Q_mpc_formation = sparse.bmat(Q_blocks, format='csc')
QN_mpc_formation = sparse.bmat(QN_blocks, format='csc')
R_mpc_formation = sparse.kron(sparse.eye(NUM_VEHICLES), R_mpc_lin_noadapt)

xmin_formation = np.tile(xmin_lin_noadapt, NUM_VEHICLES)
xmax_formation = np.tile(xmax_lin_noadapt, NUM_VEHICLES)
umin_formation = np.tile(umin_lin_noadapt, NUM_VEHICLES)
umax_formation = np.tile(umax_lin_noadapt, NUM_VEHICLES)

In [ ]:

# =========================
# Controller 实例化
# =========================
N_lin_noadapt = 80
max_iter_lin = 5

from scipy.sparse import block_diag as sparse_block_diag

A_block = sparse_block_diag([sparse.csc_matrix(A_lin_stable) for _ in range(NUM_VEHICLES)], format='csc')
B_block = sparse_block_diag([sparse.csc_matrix(B_lin) for _ in range(NUM_VEHICLES)], format='csc')
C_block = sparse_block_diag([sparse.csc_matrix(C_lin) for _ in range(NUM_VEHICLES)], format='csc')

linear_model_formation = linear_Dynamics(A_block, B_block, C_block)

solver_settings = {
    "gen_embedded_ctrl": False, "warm_start": True, "polish": True, "polish_refine_iter": 5,
    "scaling": True, "adaptive_rho": True, "check_termination": 25, "max_iter": 4000,
    "eps_abs": 1e-3, "eps_rel": 1e-3, "eps_prim_inf": 1e-3, "eps_dual_inf": 1e-3
}

controller_nmpc_formation = NonlinearMPCController(
    linear_model_formation, N_lin_noadapt, dt, umin_formation, umax_formation,
    xmin_formation, xmax_formation, Q_mpc_formation, R_mpc_formation, QN_mpc_formation, solver_settings
)

# 初始化控制器 Guess
z_init_formation = np.zeros((N_lin_noadapt + 1, n_obs_lin * NUM_VEHICLES))
u_init_formation = np.zeros((N_lin_noadapt, num_inputs * NUM_VEHICLES))
x_ref_init_2d = np.hstack([x_ref_formation, np.repeat(x_ref_formation[:, -1:], N_lin_noadapt + 1, axis=1)])[
    :, :N_lin_noadapt + 1]

controller_nmpc_formation.construct_controller(z_init=z_init_formation, u_init=u_init_formation, x_ref=x_ref_init_2d)

In [ ]:

# =========================
# Closed-loop simulation - 4 vehicles formation
# =========================
num_states_total = num_states * NUM_VEHICLES
num_inputs_total = num_inputs * NUM_VEHICLES

xt_actual_formation = np.zeros((traj_length, num_states_total))
u_formation = np.zeros((traj_length - 1, num_inputs_total))
z_formation = np.zeros((traj_length, n_obs_lin * NUM_VEHICLES))

# 初始状态赋值 (修正 v_x 初始状态以防突变)
for v in range(NUM_VEHICLES):
    idx = v * num_states
    xt_actual_formation[0, idx:idx + num_states] = np.array([
        formation_offsets[v, 0],  # s
        formation_offsets[v, 1],  # e_y
        0.0,  # e_psi
        vx_nom,  # v_x
        0.0,  # v_y
        0.0  # r
    ])

begin_formation = time.time()
fail_count = 0
terminated_early = False

# 平滑系数
ff_gain = 0.8
delta_alpha = 0.6
ax_alpha = 0.50

sys_pars_ctrl = dict(sys_pars_new)
sys_pars_ctrl.update({"vx_ref": vx_nom, "uncertainty": "NA", "amp": 0.0, "freq": 1.0,
                      "s_ref": s_ref_path, "curvature_ref": curvature_ref_path})

for k in range(traj_length - 1):
    xk_raw_total = xt_actual_formation[k, :].copy()

    for v in range(NUM_VEHICLES):
        idx = v * num_states
        xk_raw_total[idx:idx + num_states] = clip_closed_loop_state(xk_raw_total[idx:idx + num_states])

    if not is_finite_vector(xk_raw_total):
        print(f"[STOP] Non-finite state at step {k}")
        terminated_early = True
        break

    # Lift 状态
    try:
        z_current = np.zeros(n_obs_lin * NUM_VEHICLES)
        for v in range(NUM_VEHICLES):
            idx = v * num_states
            x_v_raw = xk_raw_total[idx:idx + num_states]
            x_v_scaled = standardizer_x_kdnn.transform(x_v_raw.reshape(1, -1)).flatten()
            z_current[v * n_obs_lin:(v + 1) * n_obs_lin] = lift_scaled(x_v_scaled, model_koop_dnn_lin, net_params_lin)
        z_formation[k, :] = z_current
    except Exception as e:
        print(f"[STOP] Lift failed at step {k}: {e}")
        terminated_early = True
        break

    # 【重要修复】：动态截取 MPC 参考窗口，防止末端越界报错
    end_idx = k + 1 + N_lin_noadapt + 1
    if end_idx <= x_ref_formation.shape[1]:
        xref_win = x_ref_formation[:, k + 1: end_idx]
    else:
        available_len = x_ref_formation.shape[1] - (k + 1)
        pad_len = (N_lin_noadapt + 1) - available_len
        xref_win = np.hstack([x_ref_formation[:, k + 1:], np.tile(x_ref_formation[:, -1:], (1, pad_len))])

    # MPC 求解
    try:
        controller_nmpc_formation.solve_to_convergence(
            xref_win, z_formation[k, :],
            controller_nmpc_formation.z_init,
            controller_nmpc_formation.u_init,
            max_iter=max_iter_lin, eps=1e-3
        )
        controller_nmpc_formation.update_initial_guess_()

        # 【重要修复】：严格截取第一步控制量，防止维度混乱
        u_mpc_flat = controller_nmpc_formation.cur_u.flatten()
        u_mpc = u_mpc_flat[:num_inputs_total]

    except Exception as e:
        fail_count += 1
        print(f"[WARN] MPC fallback at step {k}: {e}")
        u_mpc = u_formation[k - 1, :] if k > 0 else np.zeros(num_inputs_total)

    # 前馈、平滑与限幅
    for v in range(NUM_VEHICLES):
        idx_u = v * num_inputs
        idx_s = v * num_states
        s_current = xt_actual_formation[k, idx_s]

        idx_curv = np.argmin(np.abs(s_ref_path - s_current))
        delta_ff = ff_gain * (sys_pars["lf"] + sys_pars["lr"]) * curvature_ref_path[idx_curv]

        delta_fb = float(u_mpc[idx_u])
        ax_cmd = float(u_mpc[idx_u + 1])

        delta_cmd_raw = np.clip(delta_fb + delta_ff, umin_formation[idx_u], umax_formation[idx_u])
        ax_cmd = np.clip(ax_cmd, umin_formation[idx_u + 1], umax_formation[idx_u + 1])

        if k == 0:
            delta_applied, ax_applied = delta_cmd_raw, ax_cmd
        else:
            delta_applied = delta_alpha * u_formation[k - 1, idx_u] + (1 - delta_alpha) * delta_cmd_raw
            ax_applied = ax_alpha * u_formation[k - 1, idx_u + 1] + (1 - ax_alpha) * ax_cmd

        u_formation[k, idx_u:idx_u + 2] = [delta_applied, ax_applied]

    # 推进真实物理动力学
    for v in range(NUM_VEHICLES):
        idx = v * num_states
        u_v = u_formation[k, v * num_inputs:(v + 1) * num_inputs]
        x_v = xt_actual_formation[k, idx:idx + num_states]
        fallback = x_v.copy() if k > 0 else xt_actual_formation[0, idx:idx + num_states]

        xt_actual_formation[k + 1, idx:idx + num_states] = safe_FK_step(x_v, u_v, sys_pars_ctrl, SNR_DB, k,
                                                                        fallback_state=fallback)

end_formation = time.time()
print("\n=== Formation Simulation Completed ===")
print(f"Total steps: {traj_length - 1} | MPC fail count: {fail_count}")
print(f"Average time per step: {(end_formation - begin_formation) / max(1, traj_length - 1):.4f} s")

In [ ]:

# =========================
# Plotting - 自动截断非有限值以防绘图崩溃
# =========================
valid_len = traj_length
for idx in range(traj_length):
    if not np.all(np.isfinite(xt_actual_formation[idx, :])):
        valid_len = idx
        print(f"[WARN] 非有限状态出现在 step {idx}，绘图截断至此")
        break

# （下方为画图代码块，同你原来一致，不再赘述全贴浪费版面，可直接使用你原来的画图部分即可）

# ====================== 1. 整体路径跟踪图 ======================
plt.figure(figsize=(12, 8))
plt.plot(x_path, y_ref_path, 'k-', linewidth=3, label="Reference Path")

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
labels = ['Vehicle 0 (Leader)', 'Vehicle 1 (右前方)', 'Vehicle 2 (左后方)', 'Vehicle 3 (右后方)']

for v in range(NUM_VEHICLES):
    idx = v * num_states
    x_actual, y_actual = frenet_to_global(xt_actual_formation[:valid_len, idx],
                                          xt_actual_formation[:valid_len, idx + 1], s_ref_path, x_path, y_ref_path,
                                          psi_ref_path)
    plt.plot(x_actual, y_actual, '--', color=colors[v], linewidth=2.5, label=labels[v])

plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("四车矩形编队双移线路径跟踪", fontsize=16)
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()